# Attention Mechanism

## Start with the problem: one summary can lose the detail you need

A recurrent model compresses a prefix into a state. If a later prediction depends on one particular earlier token, a fixed summary may make retrieval difficult. Attention offers a direct, content-dependent way to combine information from available positions.

This lesson derives scaled dot-product attention, follows a small numerical example, and distinguishes attention weights from explanations or persistent memory.

## See the information flow

```text
query --------------------+
                          v
keys -> similarity scores -> softmax -> weights
                                      |
values ------------------------------+-> weighted sum -> output
```

Each query produces its own set of weights. In the two-value example below, scores $[0,1]$ produce weights about $[0.269,0.731]$ and read out about $4.924$ from values $[2,6]$.

## Turn a query into a weighted read

For an input matrix $X$, learned projections produce $Q=XW^Q$, $K=XW^K$, and $V=XW^V$. A query represents what a position seeks; keys describe candidate matches; values hold the content to combine. For one query and a matrix of keys and values:

$$s_j=\frac{q\cdot k_j}{\sqrt{d_k}},\qquad \alpha_j=\operatorname{softmax}(s)_j,\qquad o=\sum_j\alpha_jv_j.$$

The scale tempers dot-product magnitude as key dimension grows. Softmax makes nonnegative weights summing to one across unmasked choices. For a whole sequence, the corresponding operation is $\operatorname{softmax}(QK^T/\sqrt{d_k})V$: its score matrix compares every query position with every key position. The value vectors are mixed using those weights.

## Calculate a two-value attention read

Take scalar query $q=1$, keys $k_1=0$ and $k_2=1$, and values $v_1=2$, $v_2=6$. With $d_k=1$, scores are $[0,1]$. Softmax gives weights approximately $[0.269,0.731]$, so the output is $0.269(2)+0.731(6)\approx4.924$.

The query retrieves a blend, not necessarily a single value. Changing the query changes the scores and therefore the blend. In a neural network, projections and upstream representations are learned jointly with the task.

## Masks define which information may be read

A padding mask removes artificial batch-padding positions; a causal mask prevents a position from reading future tokens in autoregressive prediction. Implementations usually mask disallowed scores before softmax. An all-masked row needs special handling to avoid undefined softmax results.

Attention is not automatically interpretable: weights are one part of a computation, and changing them alone need not predict how the model output changes. Nor does attention create an unlimited external memory; its values come from representations already supplied to the layer.

**Takeaway:** attention computes a query-specific weighted read from keys and values. Correct masks and axis choices are part of the model's information-flow contract.